[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/33_beam_search.ipynb)

# 🟠 Medium: Beam Search Decoding

Implement **beam search** — the classic decoding algorithm for sequence generation.

### Signature
```python
def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token) -> list[int]:
    # log_prob_fn: takes token list, returns (V,) log-probabilities
    # Returns: best sequence (list of ints)
```

### Algorithm
1. Start with `[(0.0, [start_token])]`
2. Each step: expand each beam with top-k next tokens
3. Keep top `beam_width` beams by total log-probability
4. Stop when best beam ends with `eos_token` or `max_len` reached

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch

In [22]:
def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token):
    beams = [(0.0, [start_token])]

    for _ in range(max_len - 1):

      candidates = []
      for curr_prob, seq in beams:
        if seq[-1] == eos_token:
          continue

        logprobs = log_prob_fn(seq)
        topk = min(beam_width, logprobs.shape[-1])

        top_probs, top_toks = torch.topk(logprobs, k = topk, dim = -1)

        for p, tok in zip(top_probs, top_toks):
          candidates.append((curr_prob + p, seq + [tok.item()]))

      candidates.sort(reverse = True)
      beams = candidates[:beam_width].copy()

      if beams[0][1][-1] == eos_token:
        break

    return beams[0][1]

In [23]:
# 🧪 Debug
def simple_fn(tokens):
    lp = torch.full((5,), -10.0)
    lp[min(len(tokens), 4)] = 0.0
    return lp
seq = beam_search(simple_fn, start_token=0, max_len=5, beam_width=2, eos_token=4)
print('Sequence:', seq)

Sequence: [0, 1, 2, 3, 4]


In [24]:
# ✅ SUBMIT
from torch_judge import check
check('beam_search')


🧪 Testing: Beam Search Decoding (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Returns list starting with start_token (2.6ms)
  ✅ [2/4] Greedy path (beam=1) (0.9ms)
  ✅ [3/4] Beam finds better path than greedy (1.2ms)
  ✅ [4/4] Stops at eos (0.4ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (5.2ms total)
  Progress saved. Run status() to see your dashboard.

